# **Australian Solar Generation Forecasting (30-min intervals)**

## **Step 1: Environment Setup & Time Series Fundamentals**

**Research/Learn:**

*   **Time series forecasting** predicts future values using historical data, crucial for trend and seasonal analysis. **Univariate forecasting** uses a **single** variable's past to predict its future, while **multivariate** uses **multiple** interdependent variables. **Probabilistic forecasts** provide a **probability distribution** of potential future values, capturing uncertainty better than deterministic point forecasts.

*   **NASA POWER** (Source API docs: https://power.larc.nasa.gov/api/pages/#/) provides analysis-ready solar and meteorological time series for
energy applications. The Temporal API family includes **Hourly**, **Daily**, **Monthly**, and **Climatology** services. Of these, Hourly is the most useful for our energy optimization engine because it provides hour-by-hour values, while Daily is a strong fallback or simpler first integration. NASA’s Temporal overview also shows that Hourly supports Point only, while Daily, Monthly, and Climatology support both Point and Regional. For our system, NASA POWER is useful because it adds a real environmental layer
for:
    * solar generation modeling
    * battery charge/discharge simulation
    * EV charging optimization
    * TOU-aware cost simulation
    * stronger scenario confidence

*   **Australian postcode datasets** are geographical collections that link postcodes across Australia to their corresponding physical locations. These datasets are essential for spatial analysis and localized data retrieval in technical projects.

    * **Core Attributes:** Each record typically includes the 4-digit Australian postcode, suburb/locality names, state, and precise geographic coordinates (Latitude and Longitude).

    * **Data Formats:** These datasets are commonly available in open-standard formats such as CSV for tabular processing or GeoJSON for mapping and spatial queries.

In [10]:
# install libraries ( transformers , datasets , gluonts , chronos , pandas , requests )
!pip install transformers datasets gluonts chronos pandas requests

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.7 MB/s eta 0:00:00
  Created wheel for chronos: filename=chronos-0.3-py3-none-any.whl size=7373 sha256=5c892b42fa19ddd5c465d3e9ae5405455dfe810b5bf8f62ef1243f61ee15b03a
  Stored in directory: /root/.cache/pip/wheels/b5/9c/d5/1dbbab8b167406e7058cf3c96fa031b98ef022b893a66c634a
Successfully built chronos


In [17]:
# HF account
from huggingface_hub import HfApi

api = HfApi()

user_info = api.whoami()

user_info

{'type': 'user',
 'id': '68810e5b766eae4f6f2ac465',
 'name': 'codenhenhe',
 'fullname': 'Nguyễn Nhựt Linh',
 'email': 'linhnguyennhut07103@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1780272000,
 'isPro': False,
 'avatarUrl': '/avatars/029b932f62a41833fc86c6dac1282cda.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'solar-generation-forecasting',
   'role': 'write',
   'createdAt': '2026-05-06T03:31:37.355Z'}}}

In [12]:
# HF token
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

print(type(HF_TOKEN))

<class 'str'>


In [22]:
!curl --location 'https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=ALLSKY_SFC_SW_DWN%2CT2M&community=RE&longitude=151.21&latitude=-33.86&start=20200101&end=20200131&format=JSON'

{"type":"Feature","geometry":{"type":"Point","coordinates":[151.21,-33.86,31.72]},"properties":{"parameter":{"ALLSKY_SFC_SW_DWN":{"2020010100":0.0,"2020010101":0.0,"2020010102":0.0,"2020010103":0.0,"2020010104":0.0,"2020010105":45.55,"2020010106":174.15,"2020010107":300.5,"2020010108":527.15,"2020010109":654.8,"2020010110":904.75,"2020010111":985.45,"2020010112":994.53,"2020010113":906.45,"2020010114":769.58,"2020010115":493.6,"2020010116":316.42,"2020010117":162.82,"2020010118":27.58,"2020010119":0.0,"2020010120":0.0,"2020010121":0.0,"2020010122":0.0,"2020010123":0.0,"2020010200":0.0,"2020010201":0.0,"2020010202":0.0,"2020010203":0.0,"2020010204":0.0,"2020010205":28.65,"2020010206":96.5,"2020010207":176.4,"2020010208":272.85,"2020010209":416.5,"2020010210":548.33,"2020010211":564.88,"2020010212":606.42,"2020010213":587.5,"2020010214":480.67,"2020010215":385.58,"2020010216":263.38,"2020010217":105.2,"2020010218":19.1,"2020010219":0.0,"2020010220":0.0,"2020010221":0.0,"2020010222":0.0,"

In [23]:
import requests
import pandas as pd

def get_coordinates(postcode):
    lookup_table = {
        "2000": {"lat": -33.86, "long": 151.21},
    }
    return lookup_table.get(postcode, None)

def fetch_nasa_data(lat, long, start_date="20200101", end_date="20200131"):
    url = f"https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=ALLSKY_SFC_SW_DWN%2CT2M&community=RE&longitude={long}&latitude={lat}&start={start_date}&end={end_date}&format=JSON"

    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        df = pd.DataFrame(data['properties']['parameter'])
        return df
    else:
        return f"Error: {response.status_code}"

In [27]:
postcode_test = "2000"
coords = get_coordinates(postcode_test)

if coords:
    print(f"Fetching data for Postcode {postcode_test} (Lat: {coords['lat']}, Lon: {coords['long']})...")
    result_df = fetch_nasa_data(coords['lat'], coords['long'])
    print(result_df)
else:
    print("Postcode không tồn tại trong hệ thống test.")

Fetching data for Postcode 2000 (Lat: -33.86, Lon: 151.21)...
            ALLSKY_SFC_SW_DWN    T2M
2020010100                0.0  20.19
2020010101                0.0  19.68
2020010102                0.0  19.38
2020010103                0.0  19.20
2020010104                0.0  19.19
...                       ...    ...
2020013119                0.0  25.12
2020013120                0.0  24.83
2020013121                0.0  24.65
2020013122                0.0  24.52
2020013123                0.0  24.34

[744 rows x 2 columns]
